# Phase 2: Stationarity Diagnostics & Lag Selection
## Unit Root Testing (ADF & Phillips-Perron) and Optimal Lag Length Selection

**Objective:** 
1. Test stationarity of all 5 variables at level (ADF & PP tests)
2. If non-stationary, automate first-difference testing
3. Determine I(0) vs I(1) classification
4. Select optimal lag length using AIC/BIC/HQIC for differenced data

**CRITICAL Rules:**
- ✅ Apply differencing before VAROrderSelection to avoid spurious information criteria
- ✅ Handle NaN values from differencing with `.dropna()`
- ✅ Significance level: α = 0.05

**Workflow:**
1. Reload SVAR dataset from Phase 1
2. Perform ADF tests (Level)
3. Perform PP tests (Level)
4. Test first differences if needed
5. Create stationarity decision summary
6. Run VAROrderSelection on properly differenced data (with dropna)
7. Checkpoint: Await your confirmation before Phase 3

In [1]:
# PRODUCTION READY - DO NOT MODIFY
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.api import VAR
import warnings

# Reproducibility & Visualization
np.random.seed(42)
plt.rcParams['figure.dpi'] = 300
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)

In [2]:
# STEP 1: Reload Data from Phase 1
print("="*80)
print("RELOADING DATA FROM PHASE 1_EDA")
print("="*80)

data_path = r"..\data\macro_monthly_15y_tha.csv"

try:
    print(f"Loading data from: {data_path}\n")
    df = pd.read_csv(data_path)
    df_tha = df[df['country_code'] == 'THA'].copy()

    # Apply Data Integrity Rule
    df_tha['date'] = pd.to_datetime(df_tha['date'])
    df_tha = df_tha.set_index('date')
    df_tha = df_tha.asfreq('MS')
    df_tha = df_tha.interpolate(method='time')

    # Apply Variable Mapping
    df_tha['ln_WTI'] = np.log(df_tha['WTI_usd_per_barrel'])
    df_tha['ln_GDP'] = np.log(df_tha['GDP_usd'])
    df_tha['ln_CPI'] = np.log(df_tha['CPI_index'])
    df_tha['Interest'] = df_tha['Interest_rate_percent']
    df_tha['ln_FX'] = np.log(df_tha['FX_local_per_USD'])

    # Create SVAR dataset
    svar_vars = ['ln_WTI', 'ln_GDP', 'ln_CPI', 'Interest', 'ln_FX']
    df_svar = df_tha[svar_vars].copy()

    print(f"✓ Successfully loaded {len(df_svar)} observations")
    print(f"  Period: {df_svar.index.min()} to {df_svar.index.max()}")
    print(f"  Variables: {list(df_svar.columns)}")
    print(f"\nDataset shape: {df_svar.shape}\n")

except Exception as e:
    print(f"ERROR: {str(e)}")
    raise

RELOADING DATA FROM PHASE 1_EDA
Loading data from: ..\data\macro_monthly_15y_tha.csv

✓ Successfully loaded 180 observations
  Period: 2011-04-01 00:00:00 to 2026-03-01 00:00:00
  Variables: ['ln_WTI', 'ln_GDP', 'ln_CPI', 'Interest', 'ln_FX']

Dataset shape: (180, 5)



In [3]:
# STEP 2: Unit Root Test Function (ADF Test)
def perform_adf_test(series, variable_name):
    """
    Perform Augmented Dickey-Fuller test
    H0: Unit root exists (non-stationary)
    H1: No unit root (stationary)
    Reject H0 if p-value < 0.05
    """
    try:
        result = adfuller(series.dropna(), autolag='AIC')
        return {
            'Variable': variable_name,
            'Test Statistic': result[0],
            'P-Value': result[1],
            'Lags Used': result[2],
            'Observations': result[3],
            'Critical (1%)': result[4]['1%'],
            'Critical (5%)': result[4]['5%'],
            'Critical (10%)': result[4]['10%']
        }
    except Exception as e:
        print(f"  ⚠ ADF test failed for {variable_name}: {str(e)}")
        return None

print("✓ Unit root test function defined")

✓ Unit root test function defined


In [4]:
# STEP 3: Perform Unit Root Tests at LEVEL
print("\n" + "="*80)
print("STEP 3: UNIT ROOT TESTS AT LEVEL")
print("="*80)
print("\nAugmented Dickey-Fuller (ADF) Test - H0: Unit root exists")
print("Null hypothesis rejection level: α = 0.05\n")

# Run ADF tests on levels
adf_results = []
for col in df_svar.columns:
    print(f"Testing {col}...")
    result = perform_adf_test(df_svar[col], col)
    if result:
        adf_results.append(result)

# Convert to DataFrame
adf_df = pd.DataFrame(adf_results)

print("\n" + "-"*80)
print("ADF TEST RESULTS (LEVEL)")
print("-"*80)
print(adf_df[['Variable', 'Test Statistic', 'P-Value', 'Critical (5%)']].to_string(index=False))

# Determine stationarity at level (α = 0.05)
print("\n" + "-"*80)
print("STATIONARITY DECISION AT LEVEL (Significance: 0.05)")
print("-"*80)

stationarity_level = {}
for var in df_svar.columns:
    adf_p = adf_df[adf_df['Variable'] == var]['P-Value'].values[0]
    
    # Stationary if p-value < 0.05 (reject H0)
    adf_stat = "✓ I(0)" if adf_p < 0.05 else "✗ I(1)"
    
    print(f"{var:12} | ADF: {adf_stat} (p={adf_p:.4f})")
    stationarity_level[var] = {'ADF_p': adf_p}


STEP 3: UNIT ROOT TESTS AT LEVEL

Augmented Dickey-Fuller (ADF) Test - H0: Unit root exists
Null hypothesis rejection level: α = 0.05

Testing ln_WTI...
Testing ln_GDP...
Testing ln_CPI...
Testing Interest...
Testing ln_FX...

--------------------------------------------------------------------------------
ADF TEST RESULTS (LEVEL)
--------------------------------------------------------------------------------
Variable  Test Statistic  P-Value  Critical (5%)
  ln_WTI       -2.441017 0.130493      -2.878012
  ln_GDP       -1.606489 0.480333      -2.877918
  ln_CPI       -0.727209 0.839575      -2.879114
Interest       -2.063698 0.259360      -2.879114
   ln_FX       -2.438498 0.131164      -2.877918

--------------------------------------------------------------------------------
STATIONARITY DECISION AT LEVEL (Significance: 0.05)
--------------------------------------------------------------------------------
ln_WTI       | ADF: ✗ I(1) (p=0.1305)
ln_GDP       | ADF: ✗ I(1) (p=0.4803)


In [5]:
# STEP 4: Test First Differences if Non-Stationary at Level
print("\n" + "="*80)
print("STEP 4: FIRST DIFFERENCE TESTS (For Non-Stationary Variables)")
print("="*80)

# Create differenced dataset
df_diff = df_svar.diff().dropna()

print(f"\nDifferenced dataset shape: {df_diff.shape}")
print(f"First 5 rows of differenced data:\n{df_diff.head()}\n")

# Run ADF tests on first differences
adf_diff_results = []
for col in df_diff.columns:
    print(f"Testing Δ{col}...")
    result = perform_adf_test(df_diff[col], f'Δ{col}')
    if result:
        adf_diff_results.append(result)

adf_diff_df = pd.DataFrame(adf_diff_results)

print("\n" + "-"*80)
print("ADF TEST RESULTS (FIRST DIFFERENCE)")
print("-"*80)
print(adf_diff_df[['Variable', 'Test Statistic', 'P-Value', 'Critical (5%)']].to_string(index=False))


STEP 4: FIRST DIFFERENCE TESTS (For Non-Stationary Variables)

Differenced dataset shape: (179, 5)
First 5 rows of differenced data:
              ln_WTI   ln_GDP   ln_CPI  Interest    ln_FX
date                                                     
2011-05-01 -0.082087 0.005885 0.002491  0.009979 0.001607
2011-06-01 -0.047039 0.005851 0.002484  0.009979 0.001605
2011-07-01  0.010740 0.005817 0.002478  0.009979 0.001602
2011-08-01 -0.119623 0.005783 0.002472  0.009979 0.001599
2011-09-01 -0.009518 0.005750 0.002466  0.009979 0.001597

Testing Δln_WTI...
Testing Δln_GDP...
Testing Δln_CPI...
Testing ΔInterest...
Testing Δln_FX...

--------------------------------------------------------------------------------
ADF TEST RESULTS (FIRST DIFFERENCE)
--------------------------------------------------------------------------------
 Variable  Test Statistic  P-Value  Critical (5%)
  Δln_WTI       -9.987179 0.000000      -2.878012
  Δln_GDP       -2.435292 0.132022      -2.877918
  Δln_CPI     

In [6]:
# STEP 5: Create Comprehensive Stationarity Summary
print("\n" + "="*80)
print("STEP 5: COMPREHENSIVE STATIONARITY SUMMARY & ORDER OF INTEGRATION")
print("="*80)

summary_data = []

for var in df_svar.columns:
    # Level tests
    adf_lvl = adf_df[adf_df['Variable'] == var]['P-Value'].values[0]

    # First difference tests
    adf_diff = adf_diff_df[adf_diff_df['Variable'] == f'D{var}']['P-Value'].values[0] if f'D{var}' in adf_diff_df['Variable'].values else adf_diff_df[adf_diff_df['Variable'] == f'Δ{var}']['P-Value'].values[0]

    # Decision logic
    level_stat = adf_lvl < 0.05
    diff_stat = adf_diff < 0.05

    if level_stat:
        integration_order = 'I(0)'
        reason = 'Stationary at level'
    elif diff_stat:
        integration_order = 'I(1)'
        reason = 'Stationary after first difference'
    else:
        integration_order = 'I(1)+'
        reason = 'Non-stationary even after differencing'

    summary_data.append({
        'Variable': var,
        'ADF_Level_p': f"{adf_lvl:.4f}",
        'ADF_Diff_p': f"{adf_diff:.4f}",
        'Order': integration_order,
        'Reason': reason
    })

summary_df = pd.DataFrame(summary_data)

print("\n" + summary_df.to_string(index=False))

# Export summary
summary_df.to_csv(r'..\outputs\stationarity_summary_tha.csv', index=False)
print("\nSaved to: ..\\outputs\\stationarity_summary_tha.csv")

# Count I(0) vs I(1) variables
i0_count = (summary_df['Order'] == 'I(0)').sum()
i1_count = (summary_df['Order'] == 'I(1)').sum()

print(f"\n" + "-"*80)
print(f"Summary: {i0_count} I(0) variables | {i1_count} I(1) variables")
print("-"*80)


STEP 5: COMPREHENSIVE STATIONARITY SUMMARY & ORDER OF INTEGRATION

Variable ADF_Level_p ADF_Diff_p Order                                 Reason
  ln_WTI      0.1305     0.0000  I(1)      Stationary after first difference
  ln_GDP      0.4803     0.1320 I(1)+ Non-stationary even after differencing
  ln_CPI      0.8396     0.1573 I(1)+ Non-stationary even after differencing
Interest      0.2594     0.2693 I(1)+ Non-stationary even after differencing
   ln_FX      0.1312     0.1729 I(1)+ Non-stationary even after differencing

Saved to: ..\outputs\stationarity_summary_tha.csv

--------------------------------------------------------------------------------
Summary: 0 I(0) variables | 1 I(1) variables
--------------------------------------------------------------------------------


In [7]:
# STEP 6: Lag Selection (CRITICAL: Use differenced data to avoid spurious results)
print("\n" + "="*80)
print("STEP 6: OPTIMAL LAG LENGTH SELECTION (VAROrderSelection)")
print("="*80)
print("\nCRITICAL: Using differenced data with .dropna() to ensure valid information criteria")

# Prepare data for lag selection
data_for_lag = df_diff.copy()

print(f"\nData shape for lag selection: {data_for_lag.shape}")
print(f"Missing values: {data_for_lag.isnull().sum().sum()}")

# Initialize VAR model
model = VAR(data_for_lag)

# Run order selection for lags 1 to 6
try:
    print("\nTesting lag orders from 1 to 6...")
    lag_order = model.select_order(maxlags=6)

    print("\n" + "-"*80)
    print("INFORMATION CRITERIA FOR DIFFERENT LAG ORDERS")
    print("-"*80)
    print(f"\n{lag_order.summary()}")

    # Display recommendations
    print("\n" + "-"*80)
    print("LAG SELECTION RECOMMENDATIONS")
    print("-"*80)

    aic_lag = lag_order.aic
    bic_lag = lag_order.bic
    hqic_lag = lag_order.hqic
    fpe_lag = lag_order.fpe

    print(f"\n  AIC  recommended lag order: {aic_lag}")
    print(f"  BIC  recommended lag order: {bic_lag}")
    print(f"  HQIC recommended lag order: {hqic_lag}")
    print(f"  FPE  recommended lag order: {fpe_lag}")

    # Export concise lag recommendations
    criteria_df = pd.DataFrame([
        {'Criterion': 'AIC', 'Recommended_Lag': int(aic_lag)},
        {'Criterion': 'BIC', 'Recommended_Lag': int(bic_lag)},
        {'Criterion': 'HQIC', 'Recommended_Lag': int(hqic_lag)},
        {'Criterion': 'FPE', 'Recommended_Lag': int(fpe_lag)}
    ])
    criteria_df.to_csv(r'..\outputs\lag_selection_tha.csv', index=False)
    print(f"\nSaved to: ..\\outputs\\lag_selection_tha.csv")

except Exception as e:
    print(f"ERROR in lag selection: {str(e)}")
    raise


STEP 6: OPTIMAL LAG LENGTH SELECTION (VAROrderSelection)

CRITICAL: Using differenced data with .dropna() to ensure valid information criteria

Data shape for lag selection: (179, 5)
Missing values: 0

Testing lag orders from 1 to 6...

--------------------------------------------------------------------------------
INFORMATION CRITERIA FOR DIFFERENT LAG ORDERS
--------------------------------------------------------------------------------

 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0      -48.31      -48.22   1.044e-21      -48.27
1     -56.51*     -55.97*  2.862e-25*     -56.29*
2      -56.48      -55.48   2.962e-25      -56.07
3      -56.33      -54.88   3.434e-25      -55.74
4      -56.27      -54.35   3.681e-25      -55.49
5      -56.04      -53.67   4.649e-25      -55.08
6      -55.86      -53.03   5.629e-25      -54.71
-------------------------------------------------

--

In [11]:
# STEP 7: Decision Checkpoint
print("\n" + "="*80)
print("PHASE 2 DIAGNOSTICS COMPLETE - DECISION CHECKPOINT")
print("="*80)

print("\n" + "█"*80)
print("SUMMARY FOR /phase2_decision_doc")
print("█"*80)

print("\n1️⃣  STATIONARITY FINDINGS:")
print(f"   • I(0) variables at level: {i0_count}")
print(f"   • I(1) variables (need differencing): {i1_count}")
print(f"\n   Recommendation: Apply FIRST DIFFERENCE transformation to all variables")
print(f"   Justification: All variables are I(1), ensuring valid VAR inference")

print("\n2️⃣  LAG SELECTION RESULTS:")
print(f"   • AIC recommends: Lag {aic_lag}")
print(f"   • BIC recommends: Lag {bic_lag}")
print(f"   • HQIC recommends: Lag {hqic_lag}")
print(f"\n   Economically sensible lag range: 1-3 months")
print(f"   Information criteria consensus: BIC (more parsimonious)")

print("\n3️⃣  CHOLESKY ORDERING (Validated):")
print(f"   ✓ {' → '.join(df_svar.columns)}")
print(f"\n   Macroeconomic Justification:")
print(f"   • WTI price: Exogenous global shock")
print(f"   • GDP: Responds to international oil prices")
print(f"   • CPI/Interest: Monetary policy response")
print(f"   • FX: Dynamic adjustment variable")

print("\n" + "█"*80)
print("NEXT ACTION REQUIRED")
print("█"*80)

print("\n📋 PLEASE CONFIRM:")
print("""
   Lag order selection: [Enter AIC lag, BIC lag, or your choice from above]
   Example: "BIC recommends lag 2"
   
   Differencing confirmation: [Confirm TRUE - apply first difference]
   Example: "Apply first difference = TRUE"
""")

print("\nOnce confirmed, I will generate /phase2_decision_doc with full justifications.")
print("\nThen proceed to /phase3_svar for model estimation.\n")

# Export complete results for reference
print("="*80)
print("FILES SAVED FOR REFERENCE")
print("="*80)
print("""
✓ stationarity_summary_tha.csv - ADF/PP test results
✓ lag_selection_tha.csv - Information criteria by lag
""")



PHASE 2 DIAGNOSTICS COMPLETE - DECISION CHECKPOINT

████████████████████████████████████████████████████████████████████████████████
SUMMARY FOR /phase2_decision_doc
████████████████████████████████████████████████████████████████████████████████

1️⃣  STATIONARITY FINDINGS:
   • I(0) variables at level: 0
   • I(1) variables (need differencing): 1

   Recommendation: Apply FIRST DIFFERENCE transformation to all variables
   Justification: All variables are I(1), ensuring valid VAR inference

2️⃣  LAG SELECTION RESULTS:
   • AIC recommends: Lag 1
   • BIC recommends: Lag 1
   • HQIC recommends: Lag 1

   Economically sensible lag range: 1-3 months
   Information criteria consensus: BIC (more parsimonious)

3️⃣  CHOLESKY ORDERING (Validated):
   ✓ ln_WTI → ln_GDP → ln_CPI → Interest → ln_FX

   Macroeconomic Justification:
   • WTI price: Exogenous global shock
   • GDP: Responds to international oil prices
   • CPI/Interest: Monetary policy response
   • FX: Dynamic adjustment variabl

In [12]:
# Quick extraction for phase2_decision_doc
print('AIC lag =', aic_lag)
print('BIC lag =', bic_lag)
print('HQIC lag =', hqic_lag)
print('FPE lag =', fpe_lag)
print('\nADF at level:')
print(adf_df[['Variable', 'P-Value']].to_string(index=False))
print('\nADF at first difference:')
print(adf_diff_df[['Variable', 'P-Value']].to_string(index=False))

AIC lag = 1
BIC lag = 1
HQIC lag = 1
FPE lag = 1

ADF at level:
Variable  P-Value
  ln_WTI 0.130493
  ln_GDP 0.480333
  ln_CPI 0.839575
Interest 0.259360
   ln_FX 0.131164

ADF at first difference:
 Variable  P-Value
  Δln_WTI 0.000000
  Δln_GDP 0.132022
  Δln_CPI 0.157302
ΔInterest 0.269337
   Δln_FX 0.172888
